In [1]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the raw dataset
df = pd.read_csv('../data/raw/job_postings.csv')
print(f"Raw data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Raw data loaded: 12,217 rows, 15 columns


In [2]:
# These 6 columns are scraper system columns - they tell us nothing about jobs
# We drop them to keep only meaningful data
columns_to_drop = [
    'job_link',           # just a URL
    'last_processed_time',# when scraper ran
    'last_status',        # always "Finished NER"
    'got_summary',        # scraper flag
    'got_ner',            # scraper flag
    'is_being_worked'     # scraper flag
]

df = df.drop(columns=columns_to_drop)
print(f"After dropping system columns: {df.shape[1]} columns remaining")
print(f"Columns kept: {list(df.columns)}")

After dropping system columns: 9 columns remaining
Columns kept: ['job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type']


In [3]:
# 'first_seen' is currently stored as text (string)
# We convert it to a proper date format so we can do time-based analysis
df['first_seen'] = pd.to_datetime(df['first_seen'])

# Extract useful date parts as separate columns
df['year'] = df['first_seen'].dt.year
df['month'] = df['first_seen'].dt.month
df['month_name'] = df['first_seen'].dt.strftime('%B')  # e.g. "January"

print("Date column fixed!")
print(f"Date range: {df['first_seen'].min()} to {df['first_seen'].max()}")
print(f"\nSample dates:")
print(df['first_seen'].head())

Date column fixed!
Date range: 2024-01-12 00:00:00 to 2024-01-17 00:00:00

Sample dates:
0   2024-01-14
1   2024-01-14
2   2024-01-14
3   2024-01-12
4   2024-01-14
Name: first_seen, dtype: datetime64[us]


In [4]:
# Strip extra spaces and convert to title case for consistency
# e.g. "  senior data analyst  " becomes "Senior Data Analyst"
df['job_title'] = df['job_title'].str.strip().str.title()

# Clean company names the same way
df['company'] = df['company'].str.strip().str.title()

# Clean job location
df['job_location'] = df['job_location'].str.strip()

print("Text columns cleaned!")
print("\nSample job titles after cleaning:")
print(df['job_title'].value_counts().head(5))

Text columns cleaned!

Sample job titles after cleaning:
job_title
Senior Data Engineer     288
Senior Data Analyst      165
Data Engineer            150
Data Analyst             140
Senior Mlops Engineer    138
Name: count, dtype: int64


In [5]:
# Standardize job level labels to be consistent and readable
level_mapping = {
    'Mid senior': 'Mid-Senior',
    'Associate': 'Associate',
    'Entry level': 'Entry Level',
    'Director': 'Director',
    'Executive': 'Executive'
}

df['job_level'] = df['job_level'].map(level_mapping).fillna(df['job_level'])

print("Job levels after standardizing:")
print(df['job_level'].value_counts())

Job levels after standardizing:
job_level
Mid-Senior    10919
Associate      1298
Name: count, dtype: int64


In [6]:
# We found 1 missing job_location value during inspection
# We fill it with 'Unknown' rather than deleting the whole row
# This preserves all other data in that row

df['job_location'] = df['job_location'].fillna('Unknown')

# Verify no missing values remain
print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
job_title          0
company            0
job_location       0
first_seen         0
search_city        0
search_country     0
search_position    0
job_level          0
job_type           0
year               0
month              0
month_name         0
dtype: int64


In [7]:
# Extract country from job_location for easier geographic analysis
# Most entries are "City, ST" format for US or "City, Country" for international

def extract_country(location):
    if pd.isna(location) or location == 'Unknown':
        return 'Unknown'
    # If location contains "United Kingdom"
    if 'United Kingdom' in str(location):
        return 'United Kingdom'
    # If location contains "Canada"
    elif 'Canada' in str(location):
        return 'Canada'
    # If location contains "India"
    elif 'India' in str(location):
        return 'India'
    # If it's a "City, 2-letter-state" format — it's US
    parts = str(location).split(',')
    if len(parts) == 2 and len(parts[1].strip()) == 2:
        return 'United States'
    else:
        return 'Other'

df['country'] = df['job_location'].apply(extract_country)

print("Country breakdown:")
print(df['country'].value_counts())

Country breakdown:
country
United States     9594
United Kingdom     991
Other              946
Canada             625
India               60
Unknown              1
Name: count, dtype: int64


In [8]:
# Final look at our clean dataset
print("=== CLEANED DATASET SUMMARY ===")
print(f"Total rows    : {df.shape[0]:,}")
print(f"Total columns : {df.shape[1]}")
print(f"\nColumn names  : {list(df.columns)}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nData types:")
print(df.dtypes)

=== CLEANED DATASET SUMMARY ===
Total rows    : 12,217
Total columns : 13

Column names  : ['job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'year', 'month', 'month_name', 'country']

Missing values: 0

Data types:
job_title                     str
company                       str
job_location                  str
first_seen         datetime64[us]
search_city                   str
search_country                str
search_position               str
job_level                     str
job_type                      str
year                        int32
month                       int32
month_name                    str
country                       str
dtype: object


In [9]:
# Save the cleaned data to the cleaned folder
# index=False means don't save the row numbers as a column
df.to_csv('../data/cleaned/cleaned_job_postings.csv', index=False)

print("Cleaned dataset saved successfully!")
print("Location: data/cleaned/cleaned_job_postings.csv")
print(f"\nFinal shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Cleaned dataset saved successfully!
Location: data/cleaned/cleaned_job_postings.csv

Final shape: 12,217 rows x 13 columns
